# 研究 3：統計手法の検証 — v12（CPU、単独 Colab ノート）

固定Nの厳密な分布列挙、逐次停止時のconfidence sequence、カテゴリ再標本化を検査する。物理検証とは別。

**Colab での所要時間の目安：CPU で STATS_TIME。** **出力**：`results_v12/statistics_validation.json`（検査項目ごとの合否）。最後に `STATISTICS VALIDATION PASSED`。


In [ ]:
import types, sys
_analysis_v12=types.ModuleType("analysis_v12")
sys.modules["analysis_v12"]=_analysis_v12
exec(compile('"""v12 (statistics unchanged from v10; from v9.2): bounded-loss confidence intervals and categorical resource models.\n\nEach independent shot has infidelity in {0, 1/2, 1}. No half-failure\nPoisson likelihood is used. Models are descriptive, not confidence guarantees.\nLibrary only (embedded in the notebooks; imported by the tests); it does not run anything by itself.\n"""\nfrom dataclasses import dataclass\nimport numpy as np\nimport pandas as pd\nfrom scipy.optimize import brentq, minimize\nfrom scipy.special import betaln, expit, logsumexp, xlogy\n\nVERSION = "v12"\nKEYS = [\'window\', \'scenario\', \'d\', \'p\', \'p_idle\', \'kappa0\', \'kappa_in\', \'kappa_out\']\n\n\ndef aggregate_counts(mc):\n    """Validate raw chunks before grouping; reject duplicate or inconsistent data."""\n    required = KEYS + [\'shots\', \'sum_infidelity\', \'n_fail\', \'n_half\', \'run_id\', \'point_id\', \'chunk_id\', \'seed\', \'R0\', \'Rin\', \'Rout\']\n    if not set(required) <= set(mc.columns):\n        raise ValueError(f\'missing columns: {sorted(set(required) - set(mc.columns))}\')\n    if mc.empty or mc[required].isna().any().any():\n        raise ValueError(\'empty data or missing values\')\n    for col in [\'shots\', \'n_fail\', \'n_half\', \'chunk_id\', \'seed\']:\n        v = mc[col].to_numpy(float)\n        if not np.isfinite(v).all() or np.any(v != np.floor(v)) or np.any(v < 0):\n            raise ValueError(f\'{col} must contain nonnegative integers\')\n    if (mc.shots <= 0).any() or (mc.n_half > mc.n_fail).any() or (mc.n_fail > mc.shots).any():\n        raise ValueError(\'invalid shot/category counts\')\n    if not mc.window.isin([\'noisy\',\'quiet\']).all() or not mc.scenario.isin([\'A\',\'B\']).all():\n        raise ValueError(\'unknown scenario or window\')\n    for col in [\'p\',\'p_idle\',\'d\',\'kappa0\',\'kappa_in\',\'kappa_out\']:\n        if not np.isfinite(mc[col].to_numpy(float)).all():\n            raise ValueError(f\'nonfinite physical condition: {col}\')\n    if ((mc.p<=0)|(mc.p>1)|(mc.p_idle<0)|(mc.p_idle>1)|(mc.d<1)|(mc.d!=np.floor(mc.d))).any():\n        raise ValueError(\'invalid physical probability or size\')\n    expected_idle = np.where(mc.scenario==\'A\',mc.p,0.)\n    if not np.allclose(mc.p_idle,expected_idle,rtol=1e-12,atol=0):\n        raise ValueError(\'scenario and idle rate disagree\')\n    for rounds,kappa in [(\'R0\',\'kappa0\'),(\'Rin\',\'kappa_in\'),(\'Rout\',\'kappa_out\')]:\n        if (mc[kappa]<=0).any() or not np.array_equal(mc[rounds],np.maximum(1,np.ceil(mc[kappa]*mc.d))):\n            raise ValueError(\'round counts differ from the declared d/kappa convention\')\n    if not np.allclose(mc.sum_infidelity, mc.n_fail - .5 * mc.n_half, rtol=0, atol=1e-9):\n        raise ValueError(\'sum_infidelity disagrees with category counts\')\n    if mc.duplicated([\'run_id\', \'point_id\', \'chunk_id\']).any():\n        raise ValueError(\'duplicate chunk identity\')\n    # Re-running a chunk under a new run_id does not create independent samples.\n    if mc.duplicated(KEYS + [\'seed\']).any():\n        raise ValueError(\'reused random seed for the same physical point\')\n    g = mc.groupby(KEYS, dropna=False).agg(shots=(\'shots\', \'sum\'), s=(\'sum_infidelity\', \'sum\'),\n          n_fail=(\'n_fail\', \'sum\'), n_half=(\'n_half\', \'sum\')).reset_index()\n    g[\'n_one\'] = g.n_fail - g.n_half\n    g[\'n_zero\'] = g.shots - g.n_fail\n    g[\'eps\'] = g.s / g.shots\n    g[\'stderr\'] = np.sqrt(np.maximum((g.n_one + .25*g.n_half)/g.shots-g.eps**2, 0)/g.shots)\n    return g\n\n\ndef counts(df):\n    """Columns follow the order zero, half, one."""\n    return df[[\'n_zero\', \'n_half\', \'n_one\']].to_numpy(dtype=np.int64)\n\n\ndef bounded_mean_ci(row, cl=.95, family_size=1):\n    """Fixed-sample Chernoff/KL interval for independent losses in [0,1].\n\n    Convexity gives E exp(t X) <= 1-mu+mu exp(t). Invert the two\n    Chernoff tails with alpha/(2*family_size) per tail. Thus Bonferroni\n    covers the declared finite family simultaneously, without assuming that\n    half and full errors are independent events. Not an anytime interval.\n    """\n    n = int(row[\'shots\'])\n    mean = float(row[\'s\']) / n if n > 0 else np.nan\n    if n <= 0 or n != row[\'shots\'] or not 0 <= mean <= 1 or not 0 < cl < 1 or int(family_size) != family_size or family_size < 1:\n        raise ValueError(\'invalid interval arguments\')\n    threshold = np.log(2*family_size/(1-cl))/n\n    def kl(p):\n        return float(xlogy(mean, mean)-xlogy(mean,p)\n                     +xlogy(1-mean,1-mean)-xlogy(1-mean,1-p))\n    if mean == 0:\n        return 0., float(-np.expm1(-threshold))\n    if mean == 1:\n        return float(np.exp(-threshold)), 1.\n    # brentq\'s absolute tolerance matters for rare logical error rates.\n    lo = brentq(lambda p: kl(p)-threshold, np.nextafter(0., 1.), mean, xtol=1e-15)\n    hi = brentq(lambda p: kl(p)-threshold, mean, np.nextafter(1., 0.), xtol=1e-15)\n    return float(lo), float(hi)\n\n\ndef anytime_mean_cs(row, cl=.95, family_size=1):\n    """Beta-mixture confidence sequence for bounded [0,1] losses.\n\n    For each q, (q/mu)^X ((1-q)/(1-mu))^(1-X) has conditional\n    expectation <=1 under mean mu, by convexity in X. Mixing over a\n    Beta(1/2,1/2) q preserves the nonnegative supermartingale property.\n    Ville + Bonferroni gives simultaneous coverage at ALL stopping times\n    for the declared finite family. Fractional losses are allowed.\n    """\n    n = int(row[\'shots\'])\n    s = float(row[\'s\'])\n    if n <= 0 or n != row[\'shots\'] or not 0<=s<=n or not 0<cl<1 or int(family_size)!=family_size or family_size<1:\n        raise ValueError(\'invalid confidence sequence arguments\')\n    mean = s/n\n    constant = betaln(s+.5,n-s+.5)-betaln(.5,.5)-np.log(family_size/(1-cl))\n    def root(p):\n        return constant-xlogy(s,p)-xlogy(n-s,1-p)\n    lo = 0. if s==0 else brentq(root,np.nextafter(0.,1.),mean,xtol=1e-15)\n    hi = 1. if s==n else brentq(root,mean,np.nextafter(1.,0.),xtol=1e-15)\n    return float(lo), float(hi)\n\n\ndef bulk_design(d, p):\n    d, p = np.broadcast_arrays(np.asarray(d, float), np.asarray(p, float))\n    z, l = (d-7)/2, np.log(p/1e-4)\n    return np.stack([np.ones_like(z), z, l, z*l], axis=-1)\n\n\ndef window_design(d, p, alt=False):\n    d, p = np.broadcast_arrays(np.asarray(d, float), np.asarray(p, float))\n    cols = [np.ones_like(d), np.log(p/1e-4)]\n    if alt:\n        cols.append((d-7)/2)\n    return np.stack(cols, axis=-1)\n\n\ndef category_probabilities(bulk, d, p, window=None):\n    """Two category intensities, normalized with the zero category baseline.\n\n    quiet: r_j=exp(X theta_j); noisy: r_j=bulk_j+exp(Z phi_j).\n    P=(1,r_half,r_one)/(1+r_half+r_one). This is an explicit model,\n    not the old scalar power-law with a fictitious count distribution.\n    """\n    X = bulk_design(d, p)\n    log_rates = X @ np.asarray(bulk).reshape(2, 4).T\n    if window is not None:\n        window = np.asarray(window)\n        Z = window_design(d, p, alt=(window.size == 6))\n        log_rates = np.logaddexp(log_rates, Z @ window.reshape(2, -1).T)\n    logits = np.concatenate([np.zeros(log_rates.shape[:-1]+(1,)), log_rates], axis=-1)\n    return np.exp(logits-logsumexp(logits, axis=-1, keepdims=True))\n\n\ndef mean_infidelity(bulk, d, p, window=None):\n    pr = category_probabilities(bulk, d, p, window)\n    return .5*pr[..., 1]+pr[..., 2]\n\n\n@dataclass\nclass FitResult:\n    parameters: np.ndarray\n    diagnostics: dict\n\n\ndef _fit(df, observed=None, bulk=None, alt=False, start=None):\n    observed = counts(df) if observed is None else np.asarray(observed)\n    n = df.shots.to_numpy(float)\n    if observed.shape != (len(df), 3) or np.any(observed < 0) or not np.allclose(observed.sum(axis=1), n):\n        raise ValueError(\'multinomial counts do not sum to shots\')\n    if len(df) == 0:\n        raise ValueError(\'empty fit\')\n    X = bulk_design(df.d.values, df.p.values) if bulk is None else window_design(df.d.values, df.p.values, alt)\n    if np.linalg.matrix_rank(X) < X.shape[1]:\n        raise ValueError(\'rank deficient model design\')\n    width = X.shape[1]\n    B = None if bulk is None else bulk_design(df.d.values, df.p.values) @ np.asarray(bulk).reshape(2,4).T\n    def objective(theta):\n        W = X @ theta.reshape(2, width).T\n        eta = W if B is None else np.logaddexp(B, W)\n        logits = np.column_stack([np.zeros(len(df)), eta])\n        logpr = logits-logsumexp(logits, axis=1, keepdims=True)\n        value = -float(np.sum(observed*logpr))\n        residual = n[:,None]*np.exp(logpr[:,1:])-observed[:,1:]\n        if B is not None:\n            residual *= expit(W-B)\n        gradient = (residual.T @ X).reshape(-1)\n        return value, gradient\n    initial = np.concatenate([np.linalg.lstsq(X, np.log((observed[:, j]+.5)/(n+1)), rcond=None)[0] for j in [1,2]])\n    starts = [np.asarray(start)] if start is not None else [initial]\n    flat = np.zeros((2,width)); flat[:,0] = np.log((observed[:,1:].sum(axis=0)+.5)/(n.sum()+1))\n    starts += [flat.reshape(-1)]\n    attempts = []\n    valid = []\n    for x0 in starts:\n        opt = minimize(objective, np.clip(x0,-45,45), method=\'L-BFGS-B\', jac=True,\n                       bounds=[(-50.,50.)]*(2*width), options={\'maxiter\':2000,\'ftol\':1e-13,\'gtol\':1e-6,\'maxls\':50})\n        finite = bool(np.isfinite(opt.fun) and np.isfinite(opt.x).all())\n        attempts.append(dict(success=bool(opt.success), message=str(opt.message), nll=float(opt.fun),\n                             iterations=int(opt.nit), finite=finite))\n        if opt.success and finite:\n            valid.append(opt)\n    if not valid:\n        raise RuntimeError(f\'multinomial optimization failed: {attempts}\')\n    best = min(valid, key=lambda x:x.fun)\n    return FitResult(best.x, dict(success=True, nll=float(best.fun), attempts=attempts,\n                     boundary=bool(np.any(np.abs(best.x)>49.9)),\n                     category_counts=observed.sum(axis=0).astype(int).tolist(), rank=int(np.linalg.matrix_rank(X))))\n\n\ndef fit_bulk(df, observed=None, start=None):\n    return _fit(df, observed, start=start)\n\n\ndef fit_window(df, bulk, observed=None, alt=False, start=None):\n    return _fit(df, observed, bulk, alt, start)\n\n\ndef multinomial_sample(df, probabilities, rng):\n    return np.array([rng.multinomial(int(n), pr) for n, pr in zip(df.shots, probabilities)], dtype=np.int64)\n\n\ndef fit_models(main, quiet, scenarios, n_boot=200, seed=20261103, window_dmin=7):\n    if n_boot < 1:\n        raise ValueError(\'at least one bootstrap replication is required\')\n    fits, diagnostics = {}, []\n    rng = np.random.default_rng(seed)\n    for scenario in scenarios:\n        q = quiet[quiet.scenario == scenario]\n        noisy = main[main.scenario == scenario]\n        datasets = [noisy[noisy.d>=window_dmin], noisy[noisy.d>=5], noisy[noisy.d>=5]]\n        if q.d.nunique()<3 or q.p.nunique()<2 or q.n_fail.sum()<20:\n            raise ValueError(f\'insufficient bulk data for {scenario}\')\n        b = fit_bulk(q)\n        windows = [fit_window(ds, b.parameters, alt=(i==1)) for i,ds in enumerate(datasets)]\n        central = (b.parameters, *(x.parameters for x in windows))\n        diagnostics += [dict(scenario=scenario, replication=\'central\', component=\'bulk\', **b.diagnostics)]\n        diagnostics += [dict(scenario=scenario, replication=\'central\', component=name, **x.diagnostics)\n                        for name,x in zip([\'main\',\'altW\',\'W5\'], windows)]\n        pq = category_probabilities(b.parameters,q.d.values,q.p.values)\n        pn = [category_probabilities(b.parameters, ds.d.values, ds.p.values, w.parameters) for ds,w in zip(datasets,windows)]\n        boot = []\n        for iteration in range(n_boot):\n            bb = fit_bulk(q,multinomial_sample(q,pq,rng),start=b.parameters)\n            ws = [fit_window(ds,bb.parameters,multinomial_sample(ds,pr,rng),alt=(i==1),start=w.parameters)\n                  for i,(ds,pr,w) in enumerate(zip(datasets,pn,windows))]\n            boot.append((bb.parameters, *(x.parameters for x in ws)))\n            diagnostics += [dict(scenario=scenario,replication=iteration,component=\'bulk\',**bb.diagnostics)]\n            diagnostics += [dict(scenario=scenario,replication=iteration,component=name,**x.diagnostics)\n                            for name,x in zip([\'main\',\'altW\',\'W5\'],ws)]\n        fits[scenario] = dict(central=central,boot=boot,p_min_bulk=float(q.p.min()),\n                              d_max_data=int(noisy.d.max()),n_boot=n_boot)\n    return fits, diagnostics\n\n\ndef model_epsilon(fits, scenario, d, p, window=\'noisy\', params=None, variant=\'main\'):\n    pr = fits[scenario][\'central\'] if params is None else params\n    idx = [\'main\',\'altW\',\'W5\'].index(variant)+1\n    return float(mean_infidelity(pr[0],d,p,None if window==\'quiet\' else pr[idx]))\n\n\ndef data_decision(row, target, cl=.95, family_size=1):\n    lo, hi = anytime_mean_cs(row,cl,family_size)\n    decision = \'data_certified\' if hi<=target else \'data_rejected\' if lo>target else \'unresolved\'\n    return dict(status=decision, lower=lo, upper=hi, confidence=cl, family_size=int(family_size))\n\n\ndef write_diagnostics(path, diagnostics):\n    import json\n    from pathlib import Path\n    Path(path).write_text(json.dumps(diagnostics,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n',"analysis_v12.py","exec"),_analysis_v12.__dict__)


In [ ]:
import json,os
import numpy as np,pandas as pd
from scipy.stats import multinomial
import analysis_v12 as a10
checks={}
n=48440
lo,hi=a10.bounded_mean_ci({'shots':n,'s':0})
assert abs(hi-(-np.expm1(-np.log(40)/n)))<1e-12
assert hi>7.6e-5
checks['zero_failure_bound']=True
N=18
outcomes=np.array([(N-h-o,h,o) for h in range(N+1) for o in range(N-h+1)])
intervals=np.array([a10.bounded_mean_ci({'shots':N,'s':.5*h+o}) for _,h,o in outcomes])
coverage=[]
for ph,p1 in [(0,.01),(0,.1),(.1,0),(.15,.03),(.5,.2),(.01,.8)]:
    mu=p1+.5*ph
    keep=(intervals[:,0]<=mu)&(mu<=intervals[:,1])
    value=float(multinomial.pmf(outcomes,N,[1-ph-p1,ph,p1])[keep].sum())
    assert value>=.95-1e-12
    coverage.append(value)
checks['exact_fixed_sample_coverage']=coverage
# Enumerate Bernoulli path probabilities, pruning paths at first CS failure.
optional_failures=[]
for p in [.01,.1,.5]:
    alive=np.array([1.]);failed=0.
    for n in range(1,101):
        nxt=np.zeros(n+1);nxt[:-1]+=alive*(1-p);nxt[1:]+=alive*p
        for k in range(n+1):
            lo,hi=a10.anytime_mean_cs({'shots':n,'s':k})
            if not lo<=p<=hi:failed+=nxt[k];nxt[k]=0
        alive=nxt
    assert failed<=.05+1e-12
    optional_failures.append(float(failed))
checks['optional_stopping_failure_probabilities']=optional_failures
# Categories, rather than fictitious independent half-failures, are sampled.
grid=pd.DataFrame([(d,p,100000) for d in [3,5,7,9] for p in [1e-4,3e-4,1e-3]],columns=['d','p','shots'])
truth=np.array([-7.,-.7,1.8,.05,-6.,-.8,2.,.1])
pr=a10.category_probabilities(truth,grid.d,grid.p)
obs=a10.multinomial_sample(grid,pr,np.random.default_rng(42))
assert np.all(obs.sum(axis=1)==grid.shots)
fit=a10.fit_bulk(grid,obs)
assert fit.diagnostics['success']
assert np.max(np.abs(a10.category_probabilities(fit.parameters,grid.d,grid.p)-pr))<.005
checks['multinomial_fit']=True
os.makedirs('results_v12',exist_ok=True)
with open('results_v12/statistics_validation.json','w') as file:json.dump(checks,file,indent=2)
print(json.dumps(checks,indent=2))
print('STATISTICS VALIDATION PASSED')
